In [0]:


df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumnsWithTypeWidening") #addNewColumnsWithTypeWidening,addNewColumns,failOnNewColumns,rescue,none
    .option("cloudFiles.inferColumnTypes",True)
    # .option("cloudFiles.schemaHints", "amount string")
    # .option("cloudFiles.allowOverwrites",True)
    .option("cloudFiles.schemaLocation", "/Volumes/session_14_streaming/default/schemalocation/")
    .load("/Volumes/session_14_streaming/default/source/")
    .withColumns({
            "_load_dt": current_date(),
            "_load_dttm": current_timestamp(),
            "_file_name": col("_metadata.file_name"),
            "_file_path": col("_metadata.file_path"),
            "_file_size": col("_metadata.file_size"),
            "_file_mod": col("_metadata.file_modification_time")
        })
    )

(
    df.writeStream
    .outputMode("append")
    .option("mergeSchema",True)
    .option("checkpointLocation", "/Volumes/session_14_streaming/default/checkpoint/")
    .trigger(availableNow=True)
    .toTable("session_14_streaming.default.sink")
    )

In [0]:
spark.table("workspace.netflix.config_table").display()